# Hello Pipeline — PILOT, CHUNK 2 of 3: remaining MAGMA batch

Runs MAGMA for the **8 remaining traits** (SLE, IBD, RA, Heel BMD, Hair Colour, MS,
Lymphocyte Count, T1D), fetches chunk 1's `genes.out` files from the repo, and
pushes a combined 16-trait report. Run AFTER chunk 1 has finished and pushed.

**Setup:** same as chunk 1 (both datasets attached, `GITHUB_TOKEN`, Internet ON,
**Save Version -> Save & Run All**). Expected: ~1-2 h (T1D + Lymphocyte Count
are the big files). Then run chunk 3.

In [ ]:
import os, sys, subprocess
for pkg in ["scanpy", "statsmodels"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy as np
import pandas as pd
import scanpy as sc
print("imports ok; numpy", np.__version__, "| pandas", pd.__version__)


In [ ]:
# ---------- CONFIG (real pilot, locked 2026-11-10) ----------
SEED = 42
REAL_N_PERM = 1000           # code default scale; recalibration to 10000 is the code's own gate
RAW_BASE = "https://raw.githubusercontent.com/rifatahsanpul0k/my_research/main"
WORK = "/kaggle/working/pilot_inputs"
OUTDIR = "/kaggle/working/pilot_outputs"
REFS = "/kaggle/working/refs"
MAGMA_WORK = "/kaggle/working/magma_work"
for d in (WORK, OUTDIR, REFS, MAGMA_WORK):
    os.makedirs(d, exist_ok=True)

SECTIONS = {
    "A1": {"folder": "10x_human_lymph_node_A1",
           "emb": "10x_human_lymph_node_A1_seed42_embeddings.npy",
           "lab": "10x_human_lymph_node_A1_seed42_labels.npy"},
    "D1": {"folder": "10x_human_lymph_node_D1",
           "emb": "10x_human_lymph_node_D1_seed42_embeddings.npy",
           "lab": "10x_human_lymph_node_D1_seed42_labels.npy"},
}

import glob as _glob
def find_refs_root():
    cands = []
    for pat in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        cands += _glob.glob(pat)
    for root in cands:
        if os.path.exists(os.path.join(root, "magma.zip")) or os.path.isdir(os.path.join(root, "magma")):
            return root
    return None

REFS_IN = find_refs_root()
assert REFS_IN, "hello-pipeline-refs dataset not attached (no magma.zip or magma/ folder under /kaggle/input)"
print("refs dataset at:", REFS_IN)

def find_section_dir(folder):
    for pat in (f"/kaggle/input/**/{folder}/adata_RNA.h5ad",):
        hits = _glob.glob(pat, recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    return None

SEC_DIRS = {}
for sec, spec in SECTIONS.items():
    d = find_section_dir(spec["folder"])
    assert d, f"{spec['folder']} not found - attach pulokpulok/spatial-multiomics-6datasets-private"
    SEC_DIRS[sec] = d
    print(sec, "data:", d)

# ---------- Muse-GCN seed-42 embeddings + labels from the GitHub repo ----------
import urllib.request
EMB = {}
for sec, spec in SECTIONS.items():
    paths = {}
    for kind, name in (("E", spec["emb"]), ("C", spec["lab"])):
        dst = os.path.join(WORK, name)

        if not os.path.exists(dst):
            urllib.request.urlretrieve(f"{RAW_BASE}/embeddings/{name}", dst)
        paths[kind] = np.load(dst)
    assert paths["E"].shape[0] == paths["C"].shape[0], f"{sec}: E rows != C length"
    EMB[sec] = paths
    print(sec, "E:", paths["E"].shape, "| domains:", np.unique(paths["C"]))


In [ ]:
# ---------- GitHub checkpoint push helper (runs/hello-pilot-real) ----------
# Failures only warn: outputs also stay in /kaggle/working/pilot_outputs and in
# the Kaggle version output. Token comes from the GITHUB_TOKEN Kaggle secret.
import subprocess, base64, shutil
def push_outputs(message):
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        _hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
        _GIT = ["git", "-c", f"http.extraHeader={_hdr}"]
        _REPO = "/kaggle/working/my_research_push"
        if os.path.isdir(_REPO):
            shutil.rmtree(_REPO)
        subprocess.run(_GIT + ["clone", "--depth", "1", "--filter=blob:none", "--sparse",
                               "https://github.com/rifatahsanpul0k/my_research.git", _REPO],
                       check=True, capture_output=True, text=True)
        subprocess.run(["git", "sparse-checkout", "set", "runs/hello-pilot-real"],
                       cwd=_REPO, check=True, capture_output=True, text=True)
        _dst = os.path.join(_REPO, "runs", "hello-pilot-real")
        os.makedirs(_dst, exist_ok=True)
        for fn in os.listdir(OUTDIR):
            src = os.path.join(OUTDIR, fn)
            if os.path.isfile(src):
                shutil.copy(src, os.path.join(_dst, fn))
            elif os.path.isdir(src):
                shutil.copytree(src, os.path.join(_dst, fn), dirs_exist_ok=True)
        def _sh(*a):
            return subprocess.run(list(a), cwd=_REPO, capture_output=True, text=True)
        _sh("git", "config", "user.name", "kaggle-runner")
        _sh("git", "config", "user.email", "kaggle-runner@local")
        _sh("git", "add", "runs/hello-pilot-real")
        if _sh("git", "diff", "--cached", "--quiet").returncode != 0:
            _sh("git", "commit", "-m", message)
            subprocess.run(_GIT + ["pull", "--rebase", "origin", "main"], cwd=_REPO, capture_output=True, text=True)
            _r = subprocess.run(_GIT + ["push", "origin", "main"], cwd=_REPO, capture_output=True, text=True)
            print("pushed:", message if _r.returncode == 0 else "FAILED " + _r.stderr[:300])
        else:
            print("push: nothing new")
    except Exception as e:
        print("push skipped/failed (outputs safe in pilot_outputs):", str(e)[:300])


In [ ]:
# ---------- Chunk output folder (overrides the pilot OUTDIR) ----------
OUTDIR = "/kaggle/working/chunk2_outputs"
os.makedirs(OUTDIR, exist_ok=True)
print("chunk outputs ->", OUTDIR)


In [ ]:
# ---------- Reference bundles + MAGMA binary (zip OR Kaggle-extracted layout) ----------
import zipfile, shutil, urllib.request, stat
def _find1(pat):
    hits = _glob.glob(pat, recursive=True)
    assert hits, f"not found: {pat}"
    return hits[0]

def _maybe(pat):
    hits = _glob.glob(pat, recursive=True)
    return hits[0] if hits else None

def _unzip(zpath, dest):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(dest)

# Layout A: Kaggle already extracted the bundles (folders, no .zip files).
# Layout B: bundles still .zip -> extract into REFS (working dir).
if _maybe(os.path.join(REFS_IN, "**", "NCBI38.gene.loc")):
    print("refs layout: already extracted by Kaggle - using in place")
    SEARCH_ROOTS = [REFS_IN]
else:
    print("refs layout: zip bundles - extracting")
    for z in ("magma.zip", "gwas.zip"):
        zp = os.path.join(REFS_IN, z)
        assert os.path.exists(zp), f"{z} not found in refs dataset root {REFS_IN}"
        _unzip(zp, REFS)
    for inner, dest in (("magma_v1.10_static.zip", "magma_bin"),
                        ("NCBI38.zip", "gene_loc"),
                        ("g1000_eur.zip", "g1000")):
        _unzip(_find1(os.path.join(REFS, "**", inner)), os.path.join(REFS, dest))
    SEARCH_ROOTS = [REFS]

def _find_any(pat):
    for r in SEARCH_ROOTS:
        h = _maybe(os.path.join(r, "**", pat))
        if h:
            return h
    raise AssertionError(f"not found under {SEARCH_ROOTS}: {pat}")

# MAGMA binary: copy to the working dir so it can be made executable
# (/kaggle/input is read-only).
_bin_src = _find_any(os.path.join("magma_v1.10_static", "magma"))
MAGMA = os.path.join(REFS, "magma_bin", "magma")
os.makedirs(os.path.dirname(MAGMA), exist_ok=True)
if os.path.abspath(_bin_src) != os.path.abspath(MAGMA):
    shutil.copy(_bin_src, MAGMA)
os.chmod(MAGMA, os.stat(MAGMA).st_mode | stat.S_IEXEC)

GENE_LOC_38 = _find_any("NCBI38.gene.loc")
BFILE = _find_any("g1000_eur.bim")[:-4]  # strip .bim
assert os.path.exists(BFILE + ".bed") and os.path.exists(BFILE + ".fam"), "g1000_eur plink files incomplete"

# NCBI37.3 gene locations were not in the download bundle - fetch (0.35 MB, link verified 2026-11-10)
_ncbi37 = os.path.join(REFS, "NCBI37.3.zip")
if not os.path.exists(_ncbi37):
    urllib.request.urlretrieve("https://vu.data.surfsara.nl/index.php/s/Pj2orwuF2JYyKxq/download", _ncbi37)
_unzip(_ncbi37, os.path.join(REFS, "gene_loc"))
GENE_LOC_37 = _find1(os.path.join(REFS, "gene_loc", "**", "NCBI37.3.gene.loc"))

_r = subprocess.run([MAGMA], capture_output=True, text=True)
print("MAGMA binary responds:", (("MAGMA" in _r.stdout) or ("magma" in _r.stdout.lower())))
print("BFILE:", BFILE)
print("GENE_LOC_38:", GENE_LOC_38)
print("GENE_LOC_37:", GENE_LOC_37)
# locate the gwas folder (holds one subfolder per trait)
_cand = []
for r in SEARCH_ROOTS:
    _cand += [d for d in _glob.glob(os.path.join(r, "**", "SLE"), recursive=True) if os.path.isdir(d)]
assert _cand, "gwas/SLE folder not found"
GWAS_ROOT = os.path.dirname(_cand[0])
print("GWAS_ROOT:", GWAS_ROOT)


In [ ]:
CHUNK_TRAITS = ['SLE', 'IBD', 'RA', 'Heel_BMD', 'Hair_Colour', 'MS', 'Lymphocyte_Count', 'T1D']  # this chunk only
# ---------- Sumstats -> MAGMA inputs (snp-loc + pval with N column) ----------
# Column mappings verified against the live file headers on 2026-11-10.
# build: '38' -> NCBI38 gene locations, '37' -> NCBI37.3 (annotation uses each
# file's OWN chr/pos; gene analysis matches the EUR reference by rsID).
# MS and Hair_Colour carry no rsID column -> mapped through the g1000_eur .bim
# (chr:pos, GRCh37 - both files are GRCh37, so the mapping is build-consistent).
TRAIT_SPEC = {
 # key                    folder               file                                                              sep     rsid          chr             pos                    p / neglog10        N fixed  N col  build
 "SLE":                   ("SLE",               "bentham_2015_26502338_sle_efo0002690_1_gwas.sumstats.tsv.gz",       "\t",  "rsid",       "chrom",        "pos",                 "p",                23210,   None,  "37"),
 "RA":                    ("RA",                "GCST90132223.h.tsv.gz",                                               "\t",  "rsid",       "chromosome",   "base_pair_location",  "p_value",          97173,   None,  "38"),
 "IBD":                   ("IBD",               "28067908-GCST004131-EFO_0003767.h.tsv.gz",                              "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          59957,   None,  "38"),
 "MS":                    ("MS",                "GCST90558092.tsv",                                                    "\t",  None,         "chromosome",   "base_pair_location",  "p_value",          360004,  None,  "37"),
 "T1D":                   ("T1D",               "34012112-GCST90014023-EFO_0001359.h.tsv.gz",                            "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          520580,  None,  "38"),
 "Celiac":                ("CELIAC",            "20190752-GCST000612-EFO_0001060.h.tsv.gz",                              "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          15283,   None,  "38"),
 "DLBCL":                 ("DLBCL",             "GCST90624739.h.tsv.gz",                                               "\t",  "rsid",       "chromosome",   "base_pair_location",  "p_value",          658535,  None,  "38"),
 "FL":                    ("FL",                "GCST90624741.h.tsv.gz",                                               "\t",  "rsid",       "chromosome",   "base_pair_location",  "p_value",          657982,  None,  "38"),
 "CLL":                   ("CLL",               "GCST90624738.h.tsv.gz",                                               "\t",  "rsid",       "chromosome",   "base_pair_location",  "p_value",          1108777, None,  "38"),
 "Lymphocyte_Count":      ("LYMPHOCYTE_COUNT",  "32888493-GCST90002316-EFO_0004587.h.tsv.gz",                            "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          524923,  None,  "38"),
 "Adult_Standing_Height": ("HEIGHT",            "GIANT_HEIGHT_YENGO_2022_GWAS_SUMMARY_STATS_EUR.gz",                   "\t",  "RSID",       "CHR",          "POS",                 "P",                None,    "N",   "37"),
 "Heel_BMD":              ("HEEL_BMD",          "30598549-GCST006979-EFO_0009270.h.tsv.gz",                              "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          None,    "n",   "38"),
 "Hair_Colour":           ("HAIR_COLOUR",       "hair_panukbb_1747_blonde.tsv.bgz",                                    "\t",  None,         "chr",          "pos",                 "neglog10_pval_EUR",419469,  None,  "37"),
 "ASD":                   ("ASD",               "iPSYCH-PGC_ASD_Nov2017.gz",                                           "\t",  "SNP",        "CHR",          "BP",                  "P",                46350,   None,  "37"),
 "Cortical_Surface_Area": ("CORTICAL_SURFACE_AREA", "ENIGMA3_mixed_se_wo_Mean_Full_SurfArea_20190429.txt.gz",          "\\s+", "SNP",        "CHR",          "BP",                  "P",                None,    "N",   "37"),
 "Cortical_Thickness":    ("CORTICAL_THICKNESS",    "ENIGMA3_mixed_se_wo_Mean_Full_Thickness_20190429.txt.gz",          "\\s+", "SNP",        "CHR",          "BP",                  "P",                None,    "N",   "37"),
}

BIM = None
def load_bim():
    global BIM
    if BIM is None:
        bim = pd.read_csv(BFILE + ".bim", sep="\t", header=None,
                          names=["CHR", "SNP", "CM", "BP", "A1", "A2"], dtype={"CHR": str, "SNP": str, "BP": np.int64})
        BIM = bim.drop_duplicates(subset=["CHR", "BP"])[["CHR", "BP", "SNP"]]
    return BIM

def prep_trait(key, spec):
    folder, fname, sep, rsid_c, chr_c, pos_c, p_c, n_fixed, n_col, build = spec
    outdir = os.path.join(MAGMA_WORK, key)
    os.makedirs(outdir, exist_ok=True)
    snp_loc_f = os.path.join(outdir, key + ".snp_loc.txt")
    pval_f = os.path.join(outdir, key + ".pval.txt")
    if os.path.exists(snp_loc_f) and os.path.exists(pval_f):
        print(key, ": already prepped"); return snp_loc_f, pval_f, build
    path = os.path.join(GWAS_ROOT, folder, fname)
    if not os.path.exists(path):
        # fall back: find the file anywhere under GWAS_ROOT (folder/name drift)
        _hits = [h for h in _glob.glob(os.path.join(GWAS_ROOT, "**", fname), recursive=True)
                 if os.path.isfile(h)]
        if not _hits:
            _acc = next((t for t in fname.replace("_", "-").split("-")
                         if t.startswith("GCST")), None)
            if _acc:
                _hits = [h for h in _glob.glob(os.path.join(GWAS_ROOT, "**", "*" + _acc + "*"),
                                               recursive=True) if os.path.isfile(h)]
        if _hits:
            path = _hits[0]
            print(f"{key}: resolved by search -> {path}")
    assert os.path.exists(path), f"{key}: sumstats file missing: {path}"
    want = {chr_c, pos_c, p_c} | ({rsid_c} if rsid_c else set()) | ({n_col} if n_col else set())
    _comp = "gzip" if (fname.endswith(".gz") or fname.endswith(".bgz")) else None
    df = pd.read_csv(path, sep=sep, engine="python" if sep == "\\s+" else "c", compression=_comp,
                     usecols=lambda c: c in want or c == "low_confidence_EUR")
    df = df.rename(columns={chr_c: "CHR", pos_c: "BP", p_c: "PRAW"})
    if key == "Hair_Colour":
        if "low_confidence_EUR" in df.columns:
            df = df[df["low_confidence_EUR"].astype(str).str.lower() == "false"]
        df["P"] = np.power(10.0, -df["PRAW"].astype(float))
    else:
        df["P"] = df["PRAW"].astype(float)
    df["CHR"] = df["CHR"].astype(str).str.replace("^chr", "", regex=True)
    df["BP"] = pd.to_numeric(df["BP"], errors="coerce")
    df = df.dropna(subset=["BP", "P"])
    df["BP"] = df["BP"].astype(np.int64)
    if rsid_c:
        df["SNP"] = df[rsid_c].astype(str)
    else:
        bim = load_bim()
        df = df.merge(bim, on=["CHR", "BP"], how="inner")
        print(f"{key}: bim-mapped {len(df)} rows")
    df = df[df["SNP"].str.startswith("rs", na=False)]
    df["P"] = df["P"].clip(lower=1e-300, upper=1.0)
    df["N"] = pd.to_numeric(df[n_col], errors="coerce").fillna(n_fixed) if n_col else float(n_fixed)
    df = df.sort_values("P").drop_duplicates(subset=["SNP"], keep="first")
    df[["SNP", "CHR", "BP"]].to_csv(snp_loc_f, sep="\t", index=False, header=False)
    df[["SNP", "P", "N"]].to_csv(pval_f, sep="\t", index=False)
    print(f"{key}: {len(df)} SNPs prepped (build {build})")
    return snp_loc_f, pval_f, build

PREPPED = {}
for _k, _spec in [(k, TRAIT_SPEC[k]) for k in CHUNK_TRAITS]:
    PREPPED[_k] = prep_trait(_k, _spec)
print("prepped traits:", len(PREPPED))


In [ ]:
# ---------- MAGMA: annotate + gene analysis per trait (checkpointed) ----------
import time
MAGMA_OUT = os.path.join(OUTDIR, "magma")
os.makedirs(MAGMA_OUT, exist_ok=True)
GENES_OUT = {}
for key, (snp_loc_f, pval_f, build) in PREPPED.items():
    gene_loc = GENE_LOC_38 if build == "38" else GENE_LOC_37
    prefix = os.path.join(MAGMA_WORK, key, key)
    t0 = time.time()
    if not os.path.exists(prefix + ".genes.annot"):
        r = subprocess.run([MAGMA, "--annotate", "window=10,1.5", "--snp-loc", snp_loc_f,
                            "--gene-loc", gene_loc, "--out", prefix],
                           capture_output=True, text=True)
        assert r.returncode == 0, f"{key} annotate failed: {r.stdout[-500:]} {r.stderr[-500:]}"
    if not os.path.exists(prefix + ".genes.out"):
        r = subprocess.run([MAGMA, "--bfile", BFILE, "--pval", pval_f, "ncol=N",
                            "--gene-annot", prefix + ".genes.annot", "--out", prefix],
                           capture_output=True, text=True)
        assert r.returncode == 0, f"{key} gene analysis failed: {r.stdout[-800:]} {r.stderr[-500:]}"
    dst = os.path.join(MAGMA_OUT, key + ".genes.out")
    shutil.copy(prefix + ".genes.out", dst)
    GENES_OUT[key] = dst
    if os.path.exists(prefix + ".genes.annot"):
        os.remove(prefix + ".genes.annot")  # reclaim disk; re-annotate on rerun if needed
    print(f"{key}: genes.out ok ({(time.time()-t0)/60:.1f} min)")
print("MAGMA done:", len(GENES_OUT), "traits")
push_outputs("chunk 2: remaining MAGMA batch (8 traits)")


In [ ]:
CHUNK1_TRAITS = ['Celiac', 'Adult_Standing_Height', 'Cortical_Thickness', 'ASD', 'Cortical_Surface_Area', 'FL', 'DLBCL', 'CLL']
# ---------- Fetch chunk 1 genes.out from the repo + combined report ----------
import urllib.request
os.makedirs(MAGMA_OUT, exist_ok=True)
for key in CHUNK1_TRAITS:
    dst = os.path.join(MAGMA_OUT, key + ".genes.out")
    if not os.path.exists(dst):
        urllib.request.urlretrieve(
            f"{RAW_BASE}/runs/hello-pilot-real/magma/{key}.genes.out", dst)
    print("have:", key)
rows = []
for p in sorted(os.listdir(MAGMA_OUT)):
    if p.endswith(".genes.out"):
        with open(os.path.join(MAGMA_OUT, p)) as f:
            n = sum(1 for _ in f) - 1
        rows.append({"trait": p.replace(".genes.out", ""), "genes": n})
combined = pd.DataFrame(rows)
combined.to_csv(os.path.join(OUTDIR, "magma_combined_report.csv"), index=False)
print(combined)
assert len(combined) == 16, f"expected 16 genes.out after fetch, got {len(combined)}"
push_outputs("chunk 2: magma combined report (16 traits)")
